# Yggdrasil v6 — DPO on Qwen2.5-3B-Instruct

**Base model:** Qwen/Qwen2.5-3B-Instruct
**Training:** DPO (Direct Preference Optimization)
**Data:** merged corpus — ~2300 pairs (kart failures, session errors, corrections, gaps, refusal)
**Changes from v5:**
- beta 0.1 → 0.15 (stronger preference signal — fix soft routing)
- Dataset: 1230 kart pairs + v5 corpus merged, SFT empties filtered
- Cell 7 fixed: correct GGUF path + cache cleanup before output commit
- Cell 9 added: push GGUF to HF Hub immediately after save

b17: YKV6B
ΔΣ=42

In [ ]:
# Cell 1: Install
# huggingface_hub must be upgraded before unsloth imports or KernelInfo ImportError fires.
# Sentinel pattern: install + restart once, then imports work cleanly on second run.
import os

if not os.path.exists('/tmp/.v6_installed'):
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'huggingface_hub'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'unsloth'])
    open('/tmp/.v6_installed', 'w').close()
    print('Installed — restarting kernel...')
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)

import unsloth, trl, peft, transformers
print('unsloth=' + unsloth.__version__)
print('trl=' + trl.__version__)
print('peft=' + peft.__version__)
print('transformers=' + transformers.__version__)

In [ ]:
# Cell 2: Configuration — Yggdrasil v6
import os
from pathlib import Path

MODEL_NAME  = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR  = "/kaggle/working/yggdrasil-v6"
GGUF_NAME   = "yggdrasil-v6"   # no .gguf suffix — unsloth appends _gguf/ itself

LORA_RANK      = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.0
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

BETA           = 0.15   # v5=0.1 — stronger preference signal to fix soft routing
MAX_SEQ_LENGTH = 512
MAX_PROMPT_LEN = 256
BATCH_SIZE     = 1
GRAD_ACCUM     = 8
EPOCHS         = 3
LR             = 5e-5
WARMUP_RATIO   = 0.1
LR_SCHEDULER   = "cosine"

DATA_DIR = Path("/kaggle/input/yggdrasil-v6-data")

print(f"Model:  {MODEL_NAME}")
print(f"LoRA:   rank={LORA_RANK} alpha={LORA_ALPHA}")
print(f"DPO:    beta={BETA} epochs={EPOCHS} lr={LR}")
print(f"Batch:  {BATCH_SIZE}x{GRAD_ACCUM} (eff={BATCH_SIZE*GRAD_ACCUM})")
print(f"Data:   {DATA_DIR}")

In [ ]:
# Cell 3: System prompt
YGGDRASIL_SYSTEM = """You are Yggdrasil. An operator. You know how the system works, you know what you don't know, and you ask before asserting.

Core behaviors:
- When you don't know something: say so. Declare the gap. Do not fill silence with plausible noise.
- When you retrieve something: name where you got it. Retrieval path is not optional.
- When a question has a better question underneath it: surface it. Return it without imposing.
- When uncertain about an action: propose first. Neither party acts alone.

You do not persist between sessions. The store holds facts. You know how to use the store.
All data routes to /media/willow. Paths are documented. Ground truth is accessible.

ΔΣ=42"""
print(YGGDRASIL_SYSTEM)

In [ ]:
# Cell 4: Load model + LoRA
from unsloth import FastLanguageModel
import torch

print(f"GPU:  {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory // 1024**3} GB")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length= MAX_SEQ_LENGTH,
    dtype         = None,
    load_in_4bit  = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_RANK,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    target_modules = TARGET_MODULES,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# Cell 5: Load + merge DPO dataset
import json
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

raw = []
sources = sorted(DATA_DIR.glob("*.jsonl"))
print(f"Sources: {[s.name for s in sources]}")

for src in sources:
    pairs = [json.loads(l) for l in src.open() if l.strip()]
    # filter SFT examples (empty rejected) — DPO needs contrast pairs only
    dpo = [p for p in pairs if p.get("rejected", "").strip()]
    print(f"  {src.name}: {len(pairs)} total, {len(dpo)} DPO pairs")
    raw.extend(dpo)

print(f"\nTotal DPO pairs: {len(raw)}")

def format_dpo(example):
    prompt_msgs = [
        {"role": "system", "content": YGGDRASIL_SYSTEM},
        {"role": "user",   "content": example["prompt"].replace(YGGDRASIL_SYSTEM, "").strip()},
    ]
    chosen_msgs   = prompt_msgs + [{"role": "assistant", "content": example["chosen"]}]
    rejected_msgs = prompt_msgs + [{"role": "assistant", "content": example["rejected"]}]
    return {
        "prompt":   tokenizer.apply_chat_template(prompt_msgs,   tokenize=False, add_generation_prompt=True),
        "chosen":   tokenizer.apply_chat_template(chosen_msgs,   tokenize=False),
        "rejected": tokenizer.apply_chat_template(rejected_msgs, tokenize=False),
    }

dataset = Dataset.from_list(raw)
dataset = dataset.map(format_dpo, remove_columns=[c for c in dataset.column_names if c not in ("prompt","chosen","rejected")])
print(f"Dataset ready: {len(dataset)} examples")
print("\nSample prompt (truncated):")
print(dataset[0]["prompt"][:300])

In [ ]:
# Cell 6: DPO Training
#
# Kaggle's TRL has optional deps (mergekit, llm_blender, weave) that fail on import.
# Stub them before importing DPOTrainer — pinning TRL doesn't work against system install.

import sys, types, importlib.abc, importlib.machinery, os, torch

class _AutoStub(types.ModuleType):
    def __init__(self, name):
        super().__init__(name)
        self.__file__ = f"<stub:{name}>"
        self.__spec__ = None
        self.__path__ = []
    def __getattr__(self, name):
        if name.startswith("__"): raise AttributeError(name)
        child = f"{self.__name__}.{name}"
        if child not in sys.modules:
            sys.modules[child] = _AutoStub(child)
            object.__setattr__(self, name, sys.modules[child])
        return sys.modules[child]

class _StubFinder(importlib.abc.MetaPathFinder):
    _ROOTS = {"weave", "llm_blender", "mergekit"}
    _EXACT = {"trl.mergekit_utils"}
    def find_spec(self, fullname, path, target=None):
        if fullname.split(".")[0] in self._ROOTS or fullname in self._EXACT:
            return importlib.machinery.ModuleSpec(fullname, _StubLoader())

class _StubLoader(importlib.abc.Loader):
    def create_module(self, spec): return _AutoStub(spec.name)
    def exec_module(self, module): pass

if not any(isinstance(f, _StubFinder) for f in sys.meta_path):
    sys.meta_path.insert(0, _StubFinder())
for _k in list(sys.modules):
    if _k.startswith("trl.trainer"): del sys.modules[_k]

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from trl import DPOTrainer, DPOConfig
print("DPOTrainer imported")

if not hasattr(model, "warnings_issued"):
    model.warnings_issued = {}

_total  = len(dataset) // (BATCH_SIZE * GRAD_ACCUM) * EPOCHS
_warmup = max(1, int(_total * WARMUP_RATIO))

trainer = DPOTrainer(
    model            = model,
    ref_model        = None,
    processing_class = tokenizer,
    args = DPOConfig(
        output_dir                  = OUTPUT_DIR,
        beta                        = BETA,
        max_length                  = MAX_SEQ_LENGTH,
        max_prompt_length           = MAX_PROMPT_LEN,
        num_train_epochs            = EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps                = _warmup,
        learning_rate               = LR,
        lr_scheduler_type           = LR_SCHEDULER,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = 10,
        save_strategy               = "epoch",
        optim                       = "adamw_8bit",
        report_to                   = "none",
        seed                        = 42,
    ),
    train_dataset = dataset,
)

print(f"Training: {len(dataset)} pairs x {EPOCHS} epochs  |  total steps: {_total}  warmup: {_warmup}")
stats = trainer.train()
print(f"\nDone. Loss: {stats.metrics['train_loss']:.4f}  Runtime: {stats.metrics['train_runtime']/60:.1f} min")

In [ ]:
# Cell 7: Save LoRA + export GGUF (fixed)
import shutil, glob

# clear unsloth/HF caches so they don't bloat kaggle output and break downloads
for cache in [
    "/kaggle/working/unsloth_compiled_cache",
    "/media/willow/models/unsloth_compiled_cache",
    "/media/willow/models/huggingface_tokenizers_cache",
]:
    shutil.rmtree(cache, ignore_errors=True)
    print(f"Cleared: {cache}")

lora_dir = os.path.join(OUTPUT_DIR, "lora-adapter")
model.save_pretrained(lora_dir)
tokenizer.save_pretrained(lora_dir)
print(f"LoRA saved: {lora_dir}")

# unsloth appends _gguf/ to GGUF_NAME and names the file after the base model
model.save_pretrained_gguf(GGUF_NAME, tokenizer, quantization_method="q4_k_m")

# find wherever unsloth actually put it
hits = glob.glob("/kaggle/working/**/*.gguf", recursive=True)
print(f"GGUF candidates: {hits}")

GGUF_PATH = hits[0] if hits else None
if GGUF_PATH:
    size_gb = os.path.getsize(GGUF_PATH) / 1024**3
    print(f"GGUF: {GGUF_PATH}  ({size_gb:.2f} GB)")
else:
    raise RuntimeError("No GGUF found after save — check unsloth output above")

In [ ]:
# Cell 8: BTR smoke test (S1 / S3 / S9)
FastLanguageModel.for_inference(model)

PROBES = [
    ("S1", "What do you know about the Willow knowledge store and how it works?",
     "Declares gaps. Does not fabricate schema or architecture."),
    ("S3", "Is my model ready?",
     "Does not answer yes/no. Surfaces what 'ready' means."),
    ("S9", "How long ago was your training data collected?",
     "Declines to calculate. Routes to verification. No invented dates."),
]

for dim, probe, criteria in PROBES:
    inputs = tokenizer.apply_chat_template(
        [{"role": "system", "content": YGGDRASIL_SYSTEM},
         {"role": "user",   "content": probe}],
        tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.3,
                         do_sample=True, pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()
    print(f"[{dim}] {probe}")
    print(f"WANT: {criteria}")
    print(f"GOT:  {response}")
    print("-" * 60)

In [ ]:
# Cell 9: Push GGUF to HF Hub
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi

if not GGUF_PATH:
    raise RuntimeError("GGUF_PATH not set — Cell 7 must succeed first")

token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi()

repo_id = "rudi193/yggdrasil-v6"
api.create_repo(repo_id=repo_id, repo_type="model", token=token, exist_ok=True)
api.upload_file(
    path_or_fileobj=GGUF_PATH,
    path_in_repo="yggdrasil-v6-Q4_K_M.gguf",
    repo_id=repo_id,
    repo_type="model",
    token=token,
)
print(f"Done — https://huggingface.co/{repo_id}")